TASK 1

In [99]:
import pandas as pd
import sqlite3 as sql

In [100]:
conn = sql.connect("library.db")


In [101]:
# to show the name of tables in db

tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

print(tables)

        name
0    members
1      books
2  checkouts


In [102]:
members = pd.read_sql("SELECT * FROM members", conn)
books_db = pd.read_sql("SELECT * FROM books", conn)
checkouts = pd.read_sql("SELECT * FROM checkouts", conn)

display(members.head())
display(books_db.head())
display(checkouts.head())

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05


,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez


,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03


In [103]:
# query 1 to answer 1st how much is each member borrowing?

query1 = """
SELECT
    members.first_name || ' ' || members.last_name AS Name,
    COUNT(checkouts.book_id) AS Books_borrowed
FROM members
LEFT JOIN checkouts
    ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.first_name || ' ' || members.last_name
ORDER BY Books_borrowed DESC;
"""

result1 = pd.read_sql(query1, conn)
result1

,Name,Books_borrowed
0,Aya Wahba,25
1,Sherif Saleh,21
2,Ziad Saleh,19
3,Nour Nabil,18
4,Mostafa Fouad,18
...,...,...
75,Layla Fouad,0
76,Fares Sabry,0
77,Amir Wahba,0
78,Bassel Adel,0


In [104]:
# query 2 to answer 2nd which books match a chosen author pattern?
query2 = """
SELECT *
FROM books
WHERE author LIKE 'A%';
"""

result2 = pd.read_sql(query2, conn)
result2

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez
5,506,The Paper Boat Club,Aya Hafez


In [105]:
# query 3 to answer 3rd what are the most popular books?
query3 = """
SELECT
    books.title AS Title,
    COUNT(checkouts.book_id) AS checkouts_count
FROM books
JOIN checkouts
    ON books.book_id = checkouts.book_id
GROUP BY books.book_id, books.title
ORDER BY checkouts_count DESC
LIMIT 5;
"""

result3 = pd.read_sql(query3, conn)
result3

,Title,checkouts_count
0,The Silver Kite,57
1,Fossils and Fireflies,55
2,Circuits for Beginners,46
3,Kites Over Cairo,38
4,Storms and Sailboats,25


In [106]:
# query 4 to answer 4th who are the most active readers?
query4 = """
SELECT
    members.first_name || ' ' || members.last_name AS Name,
    COUNT(checkouts.book_id) AS Books_borrowed
FROM members
JOIN checkouts
    ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.first_name || ' ' || members.last_name
ORDER BY Books_borrowed DESC
LIMIT 10;
"""

result4 = pd.read_sql(query4, conn)
result4

,Name,Books_borrowed
0,Aya Wahba,25
1,Sherif Saleh,21
2,Ziad Saleh,19
3,Nour Nabil,18
4,Mostafa Fouad,18
5,Ahmed Shafik,17
6,Youssef Hegazy,17
7,Adam Fahmy,17
8,Reem Osman,16
9,Sara Rashad,16


In [107]:
# query 5 to answer What does a neighborhood's activity look like further back in time?
query5_neighborhoods = """
SELECT DISTINCT neighborhood
FROM members;
"""

neighborhoods = pd.read_sql(query5_neighborhoods, conn)
neighborhoods

print ("Maadi")
query5 = """
SELECT
    checkouts.*
FROM checkouts
JOIN members
    ON checkouts.member_id = members.member_id
WHERE members.neighborhood = 'Maadi'
ORDER BY checkouts.checkout_date DESC
LIMIT -1 OFFSET 10;
"""

result5 = pd.read_sql(query5, conn)
result5

Maadi


,checkout_id,member_id,book_id,checkout_date,return_date
0,9103,1003,513,2025-09-04,2025-09-25
1,9081,1017,502,2025-08-25,2025-09-17
2,9001,1008,501,2025-08-23,2025-08-28
3,9050,1003,521,2025-08-21,None
4,9085,1018,525,2025-08-19,2025-09-18
...,...,...,...,...,...
74,9046,1022,513,2024-02-22,None
75,9013,1002,501,2024-02-16,2024-02-29
76,9043,1005,505,2024-01-26,2024-02-04
77,9067,1008,507,2024-01-05,2024-01-29


In [108]:
# Now import json file

books = pd.read_json("books.json")
books.head()

,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books


In [109]:
# now import html file
summer_checkouts = pd.read_html("summer_checkouts.html")[0]
summer_checkouts.head()

,Member ID,Book ID,Checkout Date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [110]:
# make rename to summer checkouts columns to be like checkouts table
summer_checkouts = summer_checkouts.rename(columns={
    "Member ID": "member_id",
    "Book ID": "book_id",
    "Checkout Date": "checkout_date"
})
print(summer_checkouts.shape)
summer_checkouts.head()

(26, 3)


,member_id,book_id,checkout_date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [111]:
# combine between checkouts table and summer_checkouts of html
all_checkouts = pd.concat(
    [checkouts, summer_checkouts],
    ignore_index=True
)
print(all_checkouts.shape)
all_checkouts.head()

(417, 5)


,checkout_id,member_id,book_id,checkout_date,return_date
0,9263.0,1047,517,2024-10-21,2024-11-07
1,9340.0,1072,513,2025-08-24,2025-09-01
2,9231.0,1053,523,2024-02-04,2024-02-16
3,9129.0,1032,513,2025-06-21,2025-06-29
4,9370.0,1079,511,2025-11-11,2025-12-03


In [112]:
# combine between all checkouts and members table from the database
combined = all_checkouts.merge(
    members,
    on="member_id",
    how="left"
)
print(combined.shape)
combined.head()

(417, 11)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date
0,9263.0,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25
1,9340.0,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21
2,9231.0,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03
3,9129.0,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19
4,9370.0,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27


In [113]:
# combine between all till now with books table from database
combined = combined.merge(
    books_db,
    on="book_id",
    how="left"
)
print(combined.shape)
combined.head()

(417, 13)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author
0,9263.0,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati
1,9340.0,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir
2,9231.0,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry
3,9129.0,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir
4,9370.0,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar


In [114]:
 # combine all till now with books from json file
 combined = combined.merge(
    books,
    on="book_id",
    how="left"
)
print(combined.shape)
combined.head()

(417, 17)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
0,9263.0,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340.0,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231.0,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
3,9129.0,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
4,9370.0,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press


In [115]:
# put combined data in a csv file
combined.to_csv("task1_combined_data.csv", index=False)

In [116]:
# combined csv file

df = pd.read_csv("task1_combined_data.csv")
print("Shape (rows, columns):", df.shape)
df.head()


Shape (rows, columns): (417, 17)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
0,9263.0,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340.0,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231.0,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
3,9129.0,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
4,9370.0,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press


TASK 2

In [117]:
# know number of duplicates

df.duplicated().sum()

np.int64(8)

In [118]:
# show the duplicated rows

df[df.duplicated(keep=False)]

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
69,9296.0,1065,501,2024-12-15,2025-01-11,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press
73,9280.0,1054,517,2024-09-06,2024-09-15,Retaj,Fahmy,6.0,Heliopolis,Inactive,2024-03-21,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
113,9052.0,1019,501,2024-06-20,2024-06-29,Mostafa,Wahba,6.0,Maadi,Inactive,2024-07-20,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press
116,9194.0,1024,513,2024-09-17,2024-10-09,Youssef,Hegazy,8.0,Nasr City,inactive,2024-01-04,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
121,9193.0,1034,519,2024-09-15,2024-10-01,Aya,Wahba,9.0,Nasr City,Active,2025-01-22,Kites Over Cairo,Jasmine Wahdan,Friendship,157,2014.0,Nile Press
225,9296.0,1065,501,2024-12-15,2025-01-11,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press
244,9334.0,1065,507,2025-12-20,2026-01-05,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,Fossils and Fireflies,Dalia Serry,Science,160,2024.0,Nile Press
246,9180.0,1034,529,2024-07-27,2024-08-02,Aya,Wahba,9.0,Nasr City,Active,2025-01-22,Riddles of the Red Sea,Sara Tantawy,Mystery,298,NaN,Cairo Young Readers
278,9180.0,1034,529,2024-07-27,2024-08-02,Aya,Wahba,9.0,Nasr City,Active,2025-01-22,Riddles of the Red Sea,Sara Tantawy,Mystery,298,NaN,Cairo Young Readers
301,9334.0,1065,507,2025-12-20,2026-01-05,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,Fossils and Fireflies,Dalia Serry,Science,160,2024.0,Nile Press


In [119]:
# drop duplicates and make sure no duplicates

df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [120]:
# Missing values
print("Missing values :")
print()
print(df.isna().sum())
print()

# Total missing
print("Total missing values:", df.isna().sum().sum())
print()

Missing values :

checkout_id          26
member_id             0
book_id               0
checkout_date         0
return_date          91
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
join_date            11
title                 0
author                0
genre                 0
pages                 0
publication_year     33
publisher             0
dtype: int64

Total missing values: 222



In [121]:
# seeing nan columns and rows
print(df[df["return_date"].isna()][
    ["checkout_id", "member_id", "book_id", "checkout_date", "return_date"]
])

     checkout_id  member_id  book_id checkout_date return_date
7         9012.0       1010      501    2025-02-17         NaN
8         9127.0       1024      506    2025-06-13         NaN
22        9030.0       1018      507    2025-03-09         NaN
24        9369.0       1076      501    2025-06-05         NaN
26        9256.0       1059      521    2025-07-09         NaN
..           ...        ...      ...           ...         ...
412          NaN       1003      501    2025-07-08         NaN
413          NaN       1017      507    2025-07-11         NaN
414          NaN       1061      504    2025-07-06         NaN
415          NaN       1201      523    2025-07-08         NaN
416          NaN       1041      512    2025-07-06         NaN

[91 rows x 5 columns]


In [122]:
print(df[df["checkout_id"].isna()])

     checkout_id  member_id  book_id checkout_date return_date first_name  \
391          NaN       1026      522    2025-07-11         NaN       Nada   
392          NaN       1049      520    2025-07-11         NaN      Ahmed   
393          NaN       1062      525    2025-07-05         NaN      Tarek   
394          NaN       1065      520    2025-07-07         NaN       Adam   
395          NaN       1104      515    2025-07-07         NaN        NaN   
396          NaN       1009      503    2025-07-09         NaN     Hassan   
397          NaN       1063      522    2025-07-07         NaN      Layla   
398          NaN       1022      511    2025-07-12         NaN    Youssef   
399          NaN       1029      523    2025-07-09         NaN       Rana   
400          NaN       1201      509    2025-07-10         NaN        NaN   
401          NaN       1005      513    2025-07-10         NaN    Youssef   
402          NaN       1104      526    2025-07-05         NaN        NaN   

In [123]:
# seeing if these member ids containing names

df[df["first_name"].isna()][
    ["member_id", "first_name", "last_name"]
]


,member_id,first_name,last_name
395,1104,NaN,NaN
400,1201,NaN,NaN
402,1104,NaN,NaN
405,1150,NaN,NaN
415,1201,NaN,NaN


In [124]:
 # seeing if we could fill missing values from members table member ids dont dont have names
 df["first_name"] = df["first_name"].fillna(
    df["member_id"].map(members.set_index("member_id")["first_name"])
)

df["last_name"] = df["last_name"].fillna(
    df["member_id"].map(members.set_index("member_id")["last_name"])
)
print(df[df["first_name"].isna()][["member_id", "first_name", "last_name"]])

     member_id first_name last_name
395       1104        NaN       NaN
400       1201        NaN       NaN
402       1104        NaN       NaN
405       1150        NaN       NaN
415       1201        NaN       NaN


In [125]:
print (members[members["member_id"].isin([1104, 1201, 1150])])

Empty DataFrame
Columns: [member_id, first_name, last_name, grade, neighborhood, membership_status, join_date]
Index: []


In [126]:
# showing that 5 member ids ar orphan records

orphan_checkouts = df[~df["member_id"].isin(members["member_id"])]
print(orphan_checkouts)
print()
print()
print(orphan_checkouts[["member_id", "checkout_date"]])

     checkout_id  member_id  book_id checkout_date return_date first_name  \
395          NaN       1104      515    2025-07-07         NaN        NaN   
400          NaN       1201      509    2025-07-10         NaN        NaN   
402          NaN       1104      526    2025-07-05         NaN        NaN   
405          NaN       1150      530    2025-07-10         NaN        NaN   
415          NaN       1201      523    2025-07-08         NaN        NaN   

    last_name  grade neighborhood membership_status join_date  \
395       NaN    NaN          NaN               NaN       NaN   
400       NaN    NaN          NaN               NaN       NaN   
402       NaN    NaN          NaN               NaN       NaN   
405       NaN    NaN          NaN               NaN       NaN   
415       NaN    NaN          NaN               NaN       NaN   

                     title         author            genre  pages  \
395     Songs of the Oasis     Hoda Bakry           Poetry    316   
400    M

In [127]:
 #orphan records

 df[df["neighborhood"].isna()][
    ["member_id", "first_name", "last_name", "neighborhood"]
]

,member_id,first_name,last_name,neighborhood
395,1104,NaN,NaN,NaN
400,1201,NaN,NaN,NaN
402,1104,NaN,NaN,NaN
405,1150,NaN,NaN,NaN
415,1201,NaN,NaN,NaN


In [128]:
 #orphan records

df[df["membership_status"].isna()][
    ["member_id", "first_name", "last_name", "membership_status"]
]

,member_id,first_name,last_name,membership_status
395,1104,NaN,NaN,NaN
400,1201,NaN,NaN,NaN
402,1104,NaN,NaN,NaN
405,1150,NaN,NaN,NaN
415,1201,NaN,NaN,NaN


In [129]:
# 5 missing are related to orphan record and rest are missing from the source
df[df["grade"].isna()][
    ["member_id", "first_name", "last_name", "grade"]
]

,member_id,first_name,last_name,grade
0,1047,Sara,Rashad,NaN
28,1030,Reem,Osman,NaN
82,1030,Reem,Osman,NaN
114,1030,Reem,Osman,NaN
118,1037,Retaj,Kamel,NaN
120,1030,Reem,Osman,NaN
124,1030,Reem,Osman,NaN
143,1047,Sara,Rashad,NaN
144,1047,Sara,Rashad,NaN
154,1030,Reem,Osman,NaN


In [130]:
# trying to fill missing from members table but missing from the source

members[members["member_id"].isin(
    df[df["grade"].isna()]["member_id"]
)][
    ["member_id", "first_name", "last_name", "grade"]
]

,member_id,first_name,last_name,grade
12,1013,Ziad,Fouad,NaN
29,1030,Reem,Osman,NaN
36,1037,Retaj,Kamel,NaN
46,1047,Sara,Rashad,NaN


In [131]:
# 5 missing related to orphan records and the rest are missing
df[df["join_date"].isna()][
    ["member_id", "first_name", "last_name", "join_date"]
]

,member_id,first_name,last_name,join_date
90,1023,Hassan,Rashad,NaN
199,1002,Fares,Saleh,NaN
218,1039,Nada,Riad,NaN
300,1039,Nada,Riad,NaN
312,1002,Fares,Saleh,NaN
395,1104,NaN,NaN,NaN
400,1201,NaN,NaN,NaN
402,1104,NaN,NaN,NaN
404,1002,Fares,Saleh,NaN
405,1150,NaN,NaN,NaN


In [132]:
# trying to fill from the members table but missing from the source

members[members["member_id"].isin(
    df[df["join_date"].isna()]["member_id"]
)][["member_id", "first_name", "last_name", "join_date"]]

,member_id,first_name,last_name,join_date
1,1002,Fares,Saleh,None
22,1023,Hassan,Rashad,None
38,1039,Nada,Riad,None


In [133]:
df[df["publication_year"].isna()][
    ["book_id", "title", "author", "publication_year"]
]

,book_id,title,author,publication_year
8,506,The Paper Boat Club,Aya Hafez,NaN
9,506,The Paper Boat Club,Aya Hafez,NaN
59,503,The Lantern Maker,Adel Roushdy,NaN
71,529,Riddles of the Red Sea,Sara Tantawy,NaN
96,529,Riddles of the Red Sea,Sara Tantawy,NaN
106,529,Riddles of the Red Sea,Sara Tantawy,NaN
124,506,The Paper Boat Club,Aya Hafez,NaN
132,529,Riddles of the Red Sea,Sara Tantawy,NaN
152,506,The Paper Boat Club,Aya Hafez,NaN
162,529,Riddles of the Red Sea,Sara Tantawy,NaN


In [134]:
# finding books id with no publish_year
books[books["book_id"].isin(df[df["publication_year"].isna()]["book_id"])]

,book_id,genre,pages,publication_year,publisher
2,503,Historical,259,NaN,Nile Press
5,506,Friendship,183,NaN,Nile Press
28,529,Mystery,298,NaN,Cairo Young Readers


In [135]:
# trying to find publish year from books table but not found
df["publication_year"] = df["publication_year"].fillna(
    df["book_id"].map(
        books.set_index("book_id")["publication_year"]
    )
)
df["publication_year"].isna().sum()

np.int64(33)

In [136]:
# the books' ids with no publish yaer
df[df["publication_year"].isna()]["book_id"].unique()

array([506, 503, 529])

In [137]:
 # trying to find same words (objects) written with inconsistent formatting
 for col in df.select_dtypes(include="object").columns:
    print("\n", col)
    print(df[col].dropna().unique())


 checkout_date
['2024-10-21' '2025-08-24' '2024-02-04' '2025-06-21' '2025-11-11'
 '2024-03-28' '2025-02-17' '2025-06-13' '2024-04-15' '2024-07-10'
 '2025-02-10' '2025-10-05' '2025-07-27' '2024-03-04' '2024-07-11'
 '2025-12-16' '2024-06-26' '2025-11-07' '2025-04-20' '2025-01-24'
 '2025-09-17' '2025-03-09' '2024-03-25' '2025-06-05' '2025-08-06'
 '2025-07-09' '2024-02-13' '2025-01-03' '2024-10-22' '2025-10-11'
 '2024-03-17' '2025-12-28' '2025-12-08' '2025-03-20' '2024-09-02'
 '2024-09-12' '2025-05-28' '2024-04-22' '2024-06-22' '2025-03-10'
 '2024-03-06' '2025-10-02' '2024-01-09' '2025-05-25' '2024-08-25'
 '2025-07-05' '2025-01-08' '2024-03-14' '2024-01-26' '2024-08-11'
 '2025-01-26' '2025-07-11' '2025-02-03' '2025-03-06' '2024-11-22'
 '2024-10-17' '2025-03-18' '2024-06-23' '2025-04-23' '2024-11-10'
 '2024-12-02' '2024-10-24' '2025-10-14' '2025-10-13' '2025-03-24'
 '2025-05-23' '2025-07-23' '2024-12-15' '2024-07-28' '2025-02-09'
 '2024-09-06' '2025-04-28' '2024-11-26' '2025-08-23' '2024-0

In [138]:
 # Standardizing the writing style of the names of neighborhoods
df["neighborhood"] = df["neighborhood"].str.strip().str.title()
df["neighborhood"].unique()

array(['Heliopolis', 'Zamalek', 'Nasr City', 'Shubra', 'Maadi', nan],
      dtype=object)

In [139]:
# Standardizing the writing style of the membership_status words
df["membership_status"] = df["membership_status"].str.strip().str.title()
df["membership_status"].unique()

array(['Inactive', 'Active', nan], dtype=object)

Checking data if data cleaned or not

In [140]:
df.duplicated().sum()

np.int64(0)

In [141]:
df.isna().sum()

,0
checkout_id,26
member_id,0
book_id,0
checkout_date,0
return_date,91
first_name,5
last_name,5
grade,41
neighborhood,5
membership_status,5


In [142]:
print(df["neighborhood"].unique())
print(df["membership_status"].unique())

['Heliopolis' 'Zamalek' 'Nasr City' 'Shubra' 'Maadi' nan]
['Inactive' 'Active' nan]


In [143]:
# Saving cleaned data in a csv file
df.to_csv("task2_cleaned_data.csv", index=False)
df.shape

(409, 17)

In [144]:
import os
os.path.exists("task2_cleaned_data.csv")

True

In [145]:
df[~df["member_id"].isin(members["member_id"])]["member_id"].unique()

array([1104, 1201, 1150])

TASK 3 FAIRNESS REPORT

In [146]:
neighborhood_comparison = df.groupby("neighborhood").agg(
    members=("member_id", "nunique"),
    checkouts=("member_id", "size")
)

neighborhood_comparison

,members,checkouts
neighborhood,,
Heliopolis,13,87
Maadi,20,114
Nasr City,16,101
Shubra,5,34
Zamalek,11,68


In [148]:
neighborhood_comparison["member_percentage"] = (
    neighborhood_comparison["members"] / neighborhood_comparison["members"].sum() * 100
)

neighborhood_comparison

,members,checkouts,member_percentage
neighborhood,,,
Heliopolis,13,87,20.000000
Maadi,20,114,30.769231
Nasr City,16,101,24.615385
Shubra,5,34,7.692308
Zamalek,11,68,16.923077
